# 📊 Portfolio Backtesting System - Analytics Demo

**DADS 4002 - Database Systems Project**

**ระบบ Backtesting สำหรับ Portfolio Management ด้วย SQL-based Analytics**

---

## 🎯 วัตถุประสงค์

Notebook นี้เป็น **Demo** การใช้งาน Analytics Features ของระบบ โดย:
- ✅ ใช้ **SQL Stored Procedures** เป็นหลักในการวิเคราะห์ (ตามโจทย์ข้อ 3)
- ✅ แสดง **Actionable Insights** จากข้อมูล (ตามโจทย์ข้อ 5f)
- ✅ ข้อมูลจริงจาก **Yahoo Finance** (208,700+ records, 50 ETFs, 15 years)

---

## ⚠️ หมายเหตุสำคัญ

- **Notebook นี้เป็น Demo เท่านั้น** - สำหรับ Presentation และทดสอบ Features
- **ระบบหลักอยู่ที่ `main.py`** - เป็น Integrated System ตามโจทย์อาจารย์ข้อ 1 และ 5a
- ก่อนรัน Notebook นี้ ต้อง**รัน `setup_procedures.py` ก่อน** เพื่อติดตั้ง SQL Stored Procedures

## 📦 Import Libraries

In [ ]:
import mysql.connector
import pandas as pd
import warnings
from datetime import datetime
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully!")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🔌 Database Connection

เชื่อมต่อกับ MySQL Database: `portfolio_backtesting`

In [ ]:
# MySQL Configuration
MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'krittanut123456',
    'database': 'portfolio_backtesting'
}

# Connect to database
try:
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    print("✅ Connected to MySQL successfully!")
    print(f"📊 Database: {MYSQL_CONFIG['database']}")
    
    # Check database statistics
    cursor.execute("SELECT COUNT(*) as total FROM etf_master")
    etf_count = cursor.fetchone()['total']
    
    cursor.execute("SELECT COUNT(*) as total FROM benchmark_portfolios")
    portfolio_count = cursor.fetchone()['total']
    
    cursor.execute("SELECT COUNT(*) as total FROM price_history")
    price_count = cursor.fetchone()['total']
    
    print(f"\n📈 Database Statistics:")
    print(f"   - ETFs: {etf_count:,}")
    print(f"   - Portfolios: {portfolio_count:,}")
    print(f"   - Price History Records: {price_count:,}")
    
except mysql.connector.Error as e:
    print(f"❌ Connection Error: {e}")
    print("\n💡 Solution: ตรวจสอบว่า MySQL Server กำลังทำงานอยู่")

## 🔍 Check SQL Stored Procedures

ตรวจสอบว่า Stored Procedures และ Views ถูกติดตั้งแล้วหรือยัง

In [ ]:
# Check Views
cursor.execute("SHOW FULL TABLES WHERE Table_type = 'VIEW'")
views = cursor.fetchall()

print("📋 SQL Views ที่ติดตั้งแล้ว:")
if views:
    for view in views:
        print(f"   ✓ {view[list(view.keys())[0]]}")
else:
    print("   ❌ ไม่พบ Views - กรุณารัน setup_procedures.py ก่อน")

# Check Stored Procedures
cursor.execute("SHOW PROCEDURE STATUS WHERE Db = 'portfolio_backtesting'")
procedures = cursor.fetchall()

print("\n📋 SQL Stored Procedures ที่ติดตั้งแล้ว:")
if procedures:
    for proc in procedures:
        print(f"   ✓ {proc['Name']}")
else:
    print("   ❌ ไม่พบ Stored Procedures - กรุณารัน setup_procedures.py ก่อน")

if views and procedures:
    print("\n✅ ระบบพร้อมใช้งาน! สามารถรัน Analytics ได้")
else:
    print("\n⚠️  กรุณารัน: python3 setup_procedures.py")

---

# 🎯 Analytics Feature #1: Top Performers

## 🏆 หา Top 5 ETFs ที่มีผลตอบแทนดีที่สุด

**ใช้ SQL Stored Procedure:** `sp_get_top_performers`

**Parameters:**
- `p_metric` = 'sharpe' (เรียงตาม Sharpe Ratio)
- `p_top_n` = 5
- `p_start_date` = '2009-01-01'
- `p_end_date` = '2025-01-01'

In [ ]:
# Call Stored Procedure
cursor.callproc('sp_get_top_performers', ['sharpe', 5, '2009-01-01', '2025-01-01'])

# Fetch results
for result in cursor.stored_results():
    df_top = pd.DataFrame(result.fetchall())

# Display results
print("🏆 Top 5 ETFs ที่มี Sharpe Ratio สูงสุด:")
print("="*80)

if not df_top.empty:
    # Format display
    df_display = df_top.copy()
    df_display['annualized_return_pct'] = df_display['annualized_return_pct'].apply(lambda x: f"{x:.2f}%")
    df_display['annualized_volatility_pct'] = df_display['annualized_volatility_pct'].apply(lambda x: f"{x:.2f}%")
    df_display['sharpe_ratio'] = df_display['sharpe_ratio'].apply(lambda x: f"{x:.4f}")
    
    df_display.columns = ['Ticker', 'ETF Name', 'Annual Return', 'Volatility', 'Sharpe Ratio']
    display(df_display)
    
    # Actionable Insights
    top_ticker = df_top.iloc[0]['ticker_symbol']
    top_sharpe = df_top.iloc[0]['sharpe_ratio']
    top_return = df_top.iloc[0]['annualized_return_pct']
    
    print("\n💡 Actionable Insights:")
    print(f"   - {top_ticker} มี Sharpe Ratio สูงสุด ({top_sharpe:.4f})")
    print(f"   - ผลตอบแทนต่อปี: {top_return:.2f}%")
    print(f"   - 🎯 แนะนำสำหรับนักลงทุนที่ต้องการผลตอบแทนดีเมื่อปรับความเสี่ยง")
    
    if top_sharpe > 1.0:
        print(f"   - ✅ Sharpe Ratio > 1.0 = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
    else:
        print(f"   - ⚠️  Sharpe Ratio < 1.0 = ควรพิจารณาความเสี่ยงอย่างรอบคอบ")
else:
    print("❌ ไม่พบข้อมูล")

---

# 🎯 Analytics Feature #2: Portfolio Comparison

## ⚖️ เปรียบเทียบ Benchmark Portfolios ทั้งหมด

**ใช้ SQL Stored Procedure:** `sp_compare_portfolios`

**Parameters:**
- `p_start_date` = '2020-01-01'
- `p_end_date` = '2025-01-01'

In [ ]:
# Call Stored Procedure
cursor.callproc('sp_compare_portfolios', ['2020-01-01', '2025-01-01'])

# Fetch results
for result in cursor.stored_results():
    df_portfolios = pd.DataFrame(result.fetchall())

# Display results
print("⚖️  เปรียบเทียบ Benchmark Portfolios (2020-2025):")
print("="*80)

if not df_portfolios.empty:
    # Format display
    df_display = df_portfolios.copy()
    df_display['portfolio_return_pct'] = df_display['portfolio_return_pct'].apply(lambda x: f"{x:.2f}%")
    df_display['annualized_volatility_pct'] = df_display['annualized_volatility_pct'].apply(lambda x: f"{x:.2f}%")
    df_display['sharpe_ratio'] = df_display['sharpe_ratio'].apply(lambda x: f"{x:.4f}")
    
    df_display.columns = ['Portfolio', 'Risk Level', 'Annual Return', 'Volatility', 'Sharpe Ratio']
    display(df_display)
    
    # Actionable Insights
    best_idx = df_portfolios['sharpe_ratio'].idxmax()
    best_portfolio = df_portfolios.loc[best_idx, 'benchmark_name']
    best_sharpe = df_portfolios.loc[best_idx, 'sharpe_ratio']
    best_return = df_portfolios.loc[best_idx, 'portfolio_return_pct']
    best_risk = df_portfolios.loc[best_idx, 'risk_level']
    
    print("\n💡 Actionable Insights:")
    print(f"   - 🏆 Portfolio ที่ดีที่สุด: {best_portfolio}")
    print(f"   - Sharpe Ratio: {best_sharpe:.4f}")
    print(f"   - ผลตอบแทนต่อปี: {best_return:.2f}%")
    print(f"   - ระดับความเสี่ยง: {best_risk}")
    print(f"   - 🎯 แนะนำสำหรับ: นักลงทุนที่ต้องการ Risk-adjusted Return ที่ดี")
else:
    print("❌ ไม่พบข้อมูล")

---

# 🎯 Analytics Feature #3: ETF Correlation Analysis

## 🔗 วิเคราะห์ Correlation ระหว่าง 2 ETFs

**ใช้ SQL Stored Procedure:** `sp_get_etf_correlation`

**ตัวอย่าง:** SPY vs QQQ

In [ ]:
# Test correlation between SPY and QQQ
ticker1 = 'SPY'
ticker2 = 'QQQ'

print(f"🔗 ETF Correlation Analysis: {ticker1} vs {ticker2}")
print("="*80)

# Call Stored Procedure
result_args = cursor.callproc('sp_get_etf_correlation', [ticker1, ticker2, 0])
correlation = result_args[2]  # OUT parameter

if correlation is not None:
    print(f"\nCorrelation: {correlation:.4f}")
    print("\n📊 การแปลผล:")
    
    if correlation > 0.8:
        strength = "แข็งแกร่งมาก (Highly Correlated)"
        recommendation = "⚠️  ไม่แนะนำให้ถือทั้ง 2 ETFs ในพอร์ตเดียวกัน"
        reason = "มีความเสี่ยงคล้ายกันมาก ไม่ได้ช่วย Diversify"
    elif correlation > 0.5:
        strength = "แข็งแกร่งปานกลาง (Moderately Correlated)"
        recommendation = "⚠️  ควรพิจารณาสัดส่วนการถืออย่างรอบคอบ"
        reason = "มีความสัมพันธ์ปานกลาง อาจช่วย Diversify ได้บ้าง"
    elif correlation > 0:
        strength = "อ่อน (Weakly Correlated)"
        recommendation = "✅ เหมาะสำหรับถือร่วมกัน"
        reason = "ช่วย Diversify ความเสี่ยงได้ดี"
    elif correlation > -0.5:
        strength = "ติดลบเล็กน้อย (Slightly Negative)"
        recommendation = "✅ ดีมากสำหรับ Diversification"
        reason = "เคลื่อนไหวในทิศทางตรงข้าม ช่วยลดความเสี่ยง"
    else:
        strength = "ติดลบแข็งแกร่ง (Strongly Negative)"
        recommendation = "✅ ยอดเยี่ยมสำหรับ Hedging"
        reason = "เคลื่อนไหวตรงข้ามอย่างชัดเจน ลดความเสี่ยงได้มาก"
    
    print(f"   - ความสัมพันธ์: {strength}")
    print(f"\n💡 Actionable Insights:")
    print(f"   - {recommendation}")
    print(f"   - เหตุผล: {reason}")
    
    # Additional test: SPY vs AGG (should have lower correlation)
    print("\n" + "="*80)
    print("🔗 ตัวอย่างเพิ่มเติม: SPY (Stock) vs AGG (Bond)")
    print("="*80)
    
    result_args2 = cursor.callproc('sp_get_etf_correlation', ['SPY', 'AGG', 0])
    correlation2 = result_args2[2]
    
    if correlation2 is not None:
        print(f"\nCorrelation: {correlation2:.4f}")
        print("\n💡 Actionable Insights:")
        if correlation2 < 0.3:
            print("   - ✅ Correlation ต่ำมาก - เหมาะสำหรับ Portfolio Diversification")
            print("   - 🎯 นี่คือเหตุผลที่ 60/40 Portfolio (60% Stock, 40% Bond) เป็นที่นิยม")
            print("   - Bond และ Stock มีความสัมพันธ์ต่ำ ช่วยลดความเสี่ยงโดยรวม")
else:
    print("❌ ไม่พบข้อมูลหรือ ETF ไม่มีอยู่")

---

# 🎯 Analytics Feature #4: Sharpe Ratio Calculator

## 📊 คำนวณ Sharpe Ratio สำหรับ ETF

**ใช้ SQL Stored Procedure:** `sp_calculate_sharpe_ratio`

**Parameters:**
- `p_ticker_symbol` = 'SPY'
- `p_risk_free_rate` = 0.02 (2% risk-free rate)

In [ ]:
# Test Sharpe Ratio for different ETFs
test_etfs = [
    ('SPY', 'S&P 500 ETF', 0.02),
    ('QQQ', 'Nasdaq 100 ETF', 0.02),
    ('AGG', 'US Aggregate Bond ETF', 0.02),
    ('GLD', 'Gold ETF', 0.02)
]

print("📊 Sharpe Ratio Analysis for Various ETFs")
print("="*80)

results = []

for ticker, name, risk_free_rate in test_etfs:
    # Call Stored Procedure
    result_args = cursor.callproc('sp_calculate_sharpe_ratio', [ticker, risk_free_rate, 0])
    sharpe = result_args[2]  # OUT parameter
    
    if sharpe is not None:
        results.append({
            'Ticker': ticker,
            'ETF Name': name,
            'Risk-Free Rate': f"{risk_free_rate*100:.2f}%",
            'Sharpe Ratio': f"{sharpe:.4f}",
            'Rating': 'Excellent' if sharpe > 2 else 'Good' if sharpe > 1 else 'Fair' if sharpe > 0.5 else 'Poor'
        })

# Display results
df_sharpe = pd.DataFrame(results)
display(df_sharpe)

# Actionable Insights
print("\n💡 Actionable Insights:")
print("\n📈 เกณฑ์การประเมิน Sharpe Ratio:")
print("   - > 2.0  = Excellent (ยอดเยี่ยม)")
print("   - 1-2    = Good (ดี)")
print("   - 0.5-1  = Fair (พอใช้)")
print("   - < 0.5  = Poor (ควรหลีกเลี่ยง)")
print("\n🎯 คำแนะนำ:")
print("   - เลือก ETF ที่มี Sharpe Ratio สูง = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
print("   - พิจารณาร่วมกับปัจจัยอื่น เช่น Expense Ratio, Liquidity, Diversification")

---

# 🎯 Analytics Feature #5: ETF Performance View

## 📈 ดูผลตอบแทนของ ETFs ทั้งหมด

**ใช้ SQL View:** `vw_etf_performance`

In [ ]:
# Query ETF Performance View
query = """
SELECT 
    ticker_symbol,
    annualized_return_pct,
    annualized_volatility_pct,
    sharpe_ratio_approx
FROM vw_etf_performance
ORDER BY sharpe_ratio_approx DESC
LIMIT 10
"""

cursor.execute(query)
df_performance = pd.DataFrame(cursor.fetchall())

print("📈 Top 10 ETFs by Sharpe Ratio (from SQL View)")
print("="*80)

if not df_performance.empty:
    # Format display
    df_display = df_performance.copy()
    df_display['annualized_return_pct'] = df_display['annualized_return_pct'].apply(lambda x: f"{x:.2f}%" if x else 'N/A')
    df_display['annualized_volatility_pct'] = df_display['annualized_volatility_pct'].apply(lambda x: f"{x:.2f}%" if x else 'N/A')
    df_display['sharpe_ratio_approx'] = df_display['sharpe_ratio_approx'].apply(lambda x: f"{x:.4f}" if x else 'N/A')
    
    df_display.columns = ['Ticker', 'Annual Return', 'Volatility', 'Sharpe Ratio']
    display(df_display)
    
    print("\n💡 ข้อมูลนี้มาจาก SQL View: vw_etf_performance")
    print("   - คำนวณโดยใช้ SQL Window Functions (LAG, OVER)")
    print("   - ไม่ใช้ Python/pandas ในการคำนวณ")
    print("   - ตรงตามโจทย์อาจารย์ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์")
else:
    print("❌ ไม่พบข้อมูล")

---

# 🎯 Bonus: Portfolio Holdings Detail

## 📂 ดูรายละเอียดของ Portfolio

**ใช้ SQL Stored Procedure:** `sp_get_portfolio_weights`

In [ ]:
# Get portfolio list first
cursor.execute("SELECT benchmark_id, benchmark_name, risk_level FROM benchmark_portfolios ORDER BY benchmark_name LIMIT 5")
portfolios = cursor.fetchall()

print("📂 Portfolio Holdings Analysis")
print("="*80)

if portfolios:
    # Show first portfolio as example
    portfolio = portfolios[0]
    portfolio_id = portfolio['benchmark_id']
    portfolio_name = portfolio['benchmark_name']
    risk_level = portfolio['risk_level']
    
    print(f"\n🎯 Portfolio: {portfolio_name}")
    print(f"   Risk Level: {risk_level}")
    print(f"   Portfolio ID: {portfolio_id}")
    print("\n📊 Holdings:")
    
    # Call Stored Procedure
    cursor.callproc('sp_get_portfolio_weights', [portfolio_id])
    
    # Fetch results
    for result in cursor.stored_results():
        df_holdings = pd.DataFrame(result.fetchall())
    
    if not df_holdings.empty:
        # Format display
        df_display = df_holdings.copy()
        df_display['weight_pct'] = df_display['weight_pct'].apply(lambda x: f"{x:.2f}%")
        
        df_display.columns = ['Ticker', 'ETF Name', 'Asset Class', 'Weight']
        display(df_display)
        
        print("\n💡 Actionable Insights:")
        print(f"   - Total Holdings: {len(df_holdings)} ETFs")
        print(f"   - ใช้ SQL Stored Procedure: sp_get_portfolio_weights")
        print(f"   - ข้อมูลนี้ช่วยในการ Rebalancing Portfolio")
else:
    print("❌ ไม่พบข้อมูล Portfolio")

---

# 📊 Summary: SQL-based Analytics

## ✅ การตอบโจทย์อาจารย์

Notebook นี้แสดงให้เห็นว่า:

### **ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์** ✅
- ✅ ใช้ **8 Stored Procedures**:
  - `sp_get_top_performers`
  - `sp_compare_portfolios`
  - `sp_get_etf_correlation`
  - `sp_calculate_sharpe_ratio`
  - `sp_get_portfolio_weights`
  - และอีก 3 procedures

- ✅ ใช้ **3 SQL Views**:
  - `vw_weekly_returns` - คำนวณ Returns ด้วย SQL Window Functions
  - `vw_etf_performance` - วิเคราะห์ Performance
  - `vw_portfolio_summary` - สรุปข้อมูล Portfolio

- ✅ **ไม่ใช้ Python/pandas** ในการคำนวณ Analytics
  - Python ใช้เฉพาะการแสดงผลและเรียก SQL เท่านั้น
  - การคำนวณทั้งหมดทำใน SQL

### **ข้อ 5f: Actionable Insights** ✅
- ✅ ทุก Analytics Feature มี **คำแนะนำการลงทุน**
- ✅ ไม่ได้แค่แสดงตัวเลข แต่**บอกว่าควรทำอย่างไร**
- ✅ ตัวอย่าง:
  - Top Performers → รู้ว่าควรลงทุน ETF ไหน
  - Portfolio Comparison → รู้ว่า Portfolio ไหนดีที่สุด
  - Correlation → รู้ว่า ETF ไหนไม่ควรถือพร้อมกัน
  - Sharpe Ratio → รู้ว่าผลตอบแทนคุ้มค่ากับความเสี่ยงหรือไม่

---

## 🎯 Features ทั้งหมดของระบบ

นอกจาก Analytics ที่แสดงใน Notebook นี้แล้ว ระบบหลัก (`main.py`) ยังมี:

1. **ETF Management (CRUD)** - Create, Read, Update, Delete ETFs
2. **Portfolio Management (CRUD)** - จัดการ Portfolios
3. **System Utilities** - Logs, Backup, Statistics
4. **Text File Operations** - Transaction Logging, Database Backup
5. **Integrated Menu System** - Single entry point (ตามโจทย์ข้อ 1 และ 5a)

---

## 📝 หมายเหตุ

- **Notebook นี้เป็น Demo เท่านั้น** - สำหรับ Presentation
- **ระบบหลักอยู่ที่ `main.py`** - Integrated System ตามโจทย์อาจารย์
- ผู้ใช้**ไม่ต้องเปิด Jupyter, MySQL Workbench** เพียงแค่รัน `python main.py`

---

## 🚀 Next Steps

1. ปิด Notebook นี้
2. รัน `python3 main.py` เพื่อใช้งานระบบแบบบูรณาการ
3. ทดสอบทุก Feature ตาม `SYSTEM_READY_GUIDE.md`

---

**Thank you! 🎉**

## 🔌 Close Database Connection

In [ ]:
# Close connection
if 'cursor' in globals() and cursor:
    cursor.close()
if 'conn' in globals() and conn:
    conn.close()
    print("✅ Database connection closed successfully!")